In [12]:
# ============================================================
# 1. IMPORT LIBRARIES AND CONFIGURE THE TRANSFORMATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import types as T

# Use UTC consistently for timestamps
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

# Project configuration
BRONZE_PATH = "Files/bronze/synthea"
SILVER_SCHEMA = "silver"
SOURCE_SYSTEM = "Synthea CSV November 2021"

SOURCE_FILE = "encounters.csv"
SOURCE_TABLE = "encounters"
TARGET_TABLE = f"{SILVER_SCHEMA}.encounters"

# Create the Silver schema if it does not exist
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}"
)

print("Configuration completed.")
print(f"Bronze source: {BRONZE_PATH}")
print(f"Source file: {SOURCE_FILE}")
print(f"Target table: {TARGET_TABLE}")


StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 14, Finished, Available, Finished, False)

Configuration completed.
Bronze source: Files/bronze/synthea
Source file: encounters.csv
Target table: silver.encounters


In [13]:
# ============================================================
# 2. INGEST THE RAW BRONZE ENCOUNTERS CSV FILE
# ============================================================

encounter_file_path = (
    f"{BRONZE_PATH}/{SOURCE_FILE}"
)

encounters_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(encounter_file_path)
)

raw_encounter_count = (
    encounters_raw.count()
)

print(
    f"Source table: {SOURCE_TABLE}"
)

print(
    f"Source file: {encounter_file_path}"
)

print(
    f"Rows loaded: {raw_encounter_count:,}"
)

print(
    f"Columns loaded: "
    f"{len(encounters_raw.columns)}"
)

encounters_raw.printSchema()

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 15, Finished, Available, Finished, False)

Source table: encounters
Source file: Files/bronze/synthea/encounters.csv
Rows loaded: 61,459
Columns loaded: 15
root
 |-- Id: string (nullable = true)
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ORGANIZATION: string (nullable = true)
 |-- PROVIDER: string (nullable = true)
 |-- PAYER: string (nullable = true)
 |-- ENCOUNTERCLASS: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_ENCOUNTER_COST: string (nullable = true)
 |-- TOTAL_CLAIM_COST: string (nullable = true)
 |-- PAYER_COVERAGE: string (nullable = true)
 |-- REASONCODE: string (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)



In [14]:
# ============================================================
# 3. INSPECT RAW ENCOUNTER RECORDS
# ============================================================

display(
    encounters_raw.limit(10)
)

print(
    encounters_raw.columns
)

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e1d356d9-535a-48ee-8b75-8cdd53f92b3f)

['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']


In [15]:
# ============================================================
# 4. STANDARDIZE ENCOUNTER COLUMN NAMES
# ============================================================

encounter_column_mapping = {
    "Id": "encounter_id",
    "ID": "encounter_id",
    "START": "encounter_start_timestamp",
    "STOP": "encounter_end_timestamp",
    "PATIENT": "patient_id",
    "ORGANIZATION": "organization_id",
    "PROVIDER": "provider_id",
    "PAYER": "payer_id",
    "ENCOUNTERCLASS": "encounter_class",
    "CODE": "encounter_code",
    "DESCRIPTION": "encounter_description",
    "BASE_ENCOUNTER_COST": "base_encounter_cost",
    "TOTAL_CLAIM_COST": "total_claim_cost",
    "PAYER_COVERAGE": "payer_coverage",
    "REASONCODE": "reason_code",
    "REASONDESCRIPTION": "reason_description"
}


encounters_standardized = encounters_raw

for old_name in encounters_raw.columns:

    if old_name in encounter_column_mapping:

        new_name = encounter_column_mapping[
            old_name
        ]

    else:

        new_name = (
            old_name
            .strip()
            .lower()
        )

    encounters_standardized = (
        encounters_standardized
        .withColumnRenamed(
            old_name,
            new_name
        )
    )


# Stop if the renaming process creates
# duplicate column names
if (
    len(encounters_standardized.columns)
    !=
    len(set(encounters_standardized.columns))
):

    raise ValueError(
        "Duplicate column names were created "
        "while standardizing encounters."
    )


print(
    f"Encounters: "
    f"{len(encounters_raw.columns)} source columns -> "
    f"{len(encounters_standardized.columns)} "
    f"standardized columns"
)

print(
    encounters_standardized.columns
)

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 17, Finished, Available, Finished, False)

Encounters: 15 source columns -> 15 standardized columns
['encounter_id', 'encounter_start_timestamp', 'encounter_end_timestamp', 'patient_id', 'organization_id', 'provider_id', 'payer_id', 'encounter_class', 'encounter_code', 'encounter_description', 'base_encounter_cost', 'total_claim_cost', 'payer_coverage', 'reason_code', 'reason_description']


In [16]:
# ============================================================
# 5. CLEAN ENCOUNTER STRING VALUES
# ============================================================


def clean_string_values(df):

    cleaned_df = df

    for column_name in df.columns:

        cleaned_df = (
            cleaned_df
            .withColumn(
                column_name,
                F.when(
                    F.trim(
                        F.col(column_name)
                    ) == "",
                    F.lit(None)
                )
                .otherwise(
                    F.trim(
                        F.col(column_name)
                    )
                )
            )
        )

    return cleaned_df


# Record the row count before cleaning
encounter_count_before_cleaning = (
    encounters_standardized.count()
)


# Remove leading and trailing spaces
# and convert blank strings to null
encounters_cleaned = clean_string_values(
    encounters_standardized
)


# Record the row count after cleaning
encounter_count_after_cleaning = (
    encounters_cleaned.count()
)


print(
    f"Encounters: "
    f"{encounter_count_before_cleaning:,} rows "
    f"before cleaning, "
    f"{encounter_count_after_cleaning:,} rows "
    f"after cleaning"
)


# Stop if cleaning unexpectedly changes the row count
if (
    encounter_count_before_cleaning
    !=
    encounter_count_after_cleaning
):

    raise ValueError(
        "Encounter row count changed during "
        "string-value cleaning."
    )


print(
    "Encounter string cleaning and "
    "row-count reconciliation passed."
)

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 18, Finished, Available, Finished, False)

Encounters: 61,459 rows before cleaning, 61,459 rows after cleaning
Encounter string cleaning and row-count reconciliation passed.


In [17]:
# ============================================================
# 6. TRANSFORM ENCOUNTER DATA TYPES
# ============================================================

encounters_silver = (
    encounters_cleaned

    # Convert encounter start and end values
    # from strings to UTC timestamps
    .withColumn(
        "encounter_start_timestamp",
        F.to_timestamp(
            F.col("encounter_start_timestamp"),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )
    .withColumn(
        "encounter_end_timestamp",
        F.to_timestamp(
            F.col("encounter_end_timestamp"),
            "yyyy-MM-dd'T'HH:mm:ssX"
        )
    )

    # Keep encounter and reason codes as strings
    .withColumn(
        "encounter_code",
        F.col("encounter_code").cast(
            T.StringType()
        )
    )
    .withColumn(
        "reason_code",
        F.col("reason_code").cast(
            T.StringType()
        )
    )

    # Standardize encounter-class values
    .withColumn(
        "encounter_class",
        F.lower(
            F.col("encounter_class")
        )
    )

    # Convert financial fields to
    # fixed-precision decimal values
    .withColumn(
        "base_encounter_cost",
        F.col(
            "base_encounter_cost"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "total_claim_cost",
        F.col(
            "total_claim_cost"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "payer_coverage",
        F.col(
            "payer_coverage"
        ).cast(
            T.DecimalType(18, 2)
        )
    )

    # Create the encounter service date
    .withColumn(
        "encounter_date",
        F.to_date(
            F.col(
                "encounter_start_timestamp"
            )
        )
    )

    # Calculate encounter duration in minutes
    .withColumn(
        "encounter_duration_minutes",
        (
            F.col(
                "encounter_end_timestamp"
            ).cast("long")
            -
            F.col(
                "encounter_start_timestamp"
            ).cast("long")
        ) / F.lit(60.0)
    )

    # Calculate the amount not covered
    # by the payer at the encounter level
    .withColumn(
        "patient_responsibility",
        (
            F.col("total_claim_cost")
            -
            F.col("payer_coverage")
        ).cast(
            T.DecimalType(18, 2)
        )
    )
)


encounters_silver.printSchema()

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 19, Finished, Available, Finished, False)

root
 |-- encounter_id: string (nullable = true)
 |-- encounter_start_timestamp: timestamp (nullable = true)
 |-- encounter_end_timestamp: timestamp (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- organization_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- payer_id: string (nullable = true)
 |-- encounter_class: string (nullable = true)
 |-- encounter_code: string (nullable = true)
 |-- encounter_description: string (nullable = true)
 |-- base_encounter_cost: decimal(18,2) (nullable = true)
 |-- total_claim_cost: decimal(18,2) (nullable = true)
 |-- payer_coverage: decimal(18,2) (nullable = true)
 |-- reason_code: string (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- encounter_date: date (nullable = true)
 |-- encounter_duration_minutes: double (nullable = true)
 |-- patient_responsibility: decimal(18,2) (nullable = true)



In [18]:
# ============================================================
# 7.1 VALIDATE ENCOUNTER-TO-PATIENT RELATIONSHIPS
# ============================================================

silver_patients = (
    spark.table(
        f"{SILVER_SCHEMA}.patients"
    )
    .select(
        F.col("id").alias(
            "valid_patient_id"
        )
    )
    .distinct()
)


orphan_encounter_patients = (
    encounters_silver
    .select(
        "encounter_id",
        "patient_id"
    )
    .join(
        silver_patients,
        encounters_silver["patient_id"]
        ==
        silver_patients["valid_patient_id"],
        "left_anti"
    )
)


orphan_patient_count = (
    orphan_encounter_patients.count()
)


print(
    f"Encounter records with an invalid "
    f"patient relationship: "
    f"{orphan_patient_count:,}"
)


if orphan_patient_count > 0:

    display(
        orphan_encounter_patients.limit(20)
    )

    raise ValueError(
        "Encounter-to-patient foreign-key "
        "validation failed."
    )


print(
    "Encounter-to-patient foreign-key "
    "validation passed."
)

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 20, Finished, Available, Finished, False)

Encounter records with an invalid patient relationship: 0
Encounter-to-patient foreign-key validation passed.


In [19]:
# ============================================================
# 8. CHECK FOR DUPLICATE ENCOUNTER RECORDS
# ============================================================

encounter_row_count = (
    encounters_silver.count()
)

distinct_encounter_row_count = (
    encounters_silver
    .dropDuplicates()
    .count()
)

exact_duplicate_count = (
    encounter_row_count
    -
    distinct_encounter_row_count
)


duplicate_encounter_id_count = (
    encounters_silver
    .groupBy(
        "encounter_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


print(
    f"Exact duplicate encounter rows: "
    f"{exact_duplicate_count:,}"
)

print(
    f"Duplicated encounter IDs: "
    f"{duplicate_encounter_id_count:,}"
)


if exact_duplicate_count > 0:

    raise ValueError(
        "Exact duplicate encounter rows "
        "were detected."
    )


if duplicate_encounter_id_count > 0:

    raise ValueError(
        "The encounter primary key is "
        "not unique."
    )


print(
    "Encounter duplicate and primary-key "
    "validations passed."
)

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 21, Finished, Available, Finished, False)

Exact duplicate encounter rows: 0
Duplicated encounter IDs: 0
Encounter duplicate and primary-key validations passed.


In [20]:
# ============================================================
# 9. ADD RECORD HASH AND SILVER PROCESSING METADATA
# ============================================================

# Source-derived columns included in the row hash
encounter_hash_columns = [
    "encounter_id",
    "encounter_start_timestamp",
    "encounter_end_timestamp",
    "patient_id",
    "organization_id",
    "provider_id",
    "payer_id",
    "encounter_class",
    "encounter_code",
    "encounter_description",
    "base_encounter_cost",
    "total_claim_cost",
    "payer_coverage",
    "reason_code",
    "reason_description"
]


# Convert each value to text and represent nulls
# consistently before calculating the hash
hash_input_columns = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>")
    )
    for column_name in encounter_hash_columns
]


encounters_silver = (
    encounters_silver

    # Create a repeatable SHA-256 fingerprint
    # for each encounter record
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *hash_input_columns
            ),
            256
        )
    )

    # Add source lineage
    .withColumn(
        "source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "source_table",
        F.lit(SOURCE_TABLE)
    )
    .withColumn(
        "source_file",
        F.lit(SOURCE_FILE)
    )

    # Record when the Silver record was processed
    .withColumn(
        "silver_processed_timestamp",
        F.current_timestamp()
    )
)


print(
    "Record hash and Silver processing "
    "metadata added to encounters."
)

encounters_silver.select(
    "encounter_id",
    "record_hash",
    "source_system",
    "source_table",
    "source_file",
    "silver_processed_timestamp"
).show(
    5,
    truncate=False
)

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 22, Finished, Available, Finished, False)

Record hash and Silver processing metadata added to encounters.
+------------------------------------+----------------------------------------------------------------+-------------------------+------------+--------------+--------------------------+
|encounter_id                        |record_hash                                                     |source_system            |source_table|source_file   |silver_processed_timestamp|
+------------------------------------+----------------------------------------------------------------+-------------------------+------------+--------------+--------------------------+
|748f8357-6cc7-551d-f31a-32fa2cf84126|a992beca2d04179973bfe8515c40aa53af56404a74cfa0695d87b306c4dd8108|Synthea CSV November 2021|encounters  |encounters.csv|2026-10-21 03:45:22.102707|
|5a4735ae-423f-6563-28ab-b3d11b49b2d4|9a18e6bed74e8da732f86b6c783d056dbdac0e00b3be1c0c6598db1e22dae290|Synthea CSV November 2021|encounters  |encounters.csv|2026-10-21 03:45:22.102707|
|0bee1ce6-3

In [21]:
# ============================================================
# 10. WRITE THE ENCOUNTERS TABLE TO THE SILVER LAYER
# ============================================================

(
    encounters_silver.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Silver table written successfully: "
    f"{TARGET_TABLE}"
)

StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 23, Finished, Available, Finished, False)

Silver table written successfully: silver.encounters


In [22]:
# ============================================================
# 11. VERIFY THE SAVED SILVER ENCOUNTERS TABLE
# ============================================================

saved_silver_encounters = (
    spark.table(
        TARGET_TABLE
    )
)


# Calculate saved-table validation metrics
saved_encounter_validation = (
    saved_silver_encounters
    .agg(
        F.count("*").alias(
            "saved_row_count"
        ),

        F.countDistinct(
            "encounter_id"
        ).alias(
            "distinct_encounter_ids"
        ),

        F.sum(
            F.when(
                F.col("encounter_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_encounter_ids"
        ),

        F.sum(
            F.when(
                F.col("record_hash").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_record_hashes"
        ),

        F.sum(
            F.when(
                F.col(
                    "silver_processed_timestamp"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_processing_timestamps"
        )
    )
)


validation_result = (
    saved_encounter_validation.first()
)


saved_encounter_count = (
    validation_result[
        "saved_row_count"
    ]
)

saved_distinct_encounter_count = (
    validation_result[
        "distinct_encounter_ids"
    ]
)

null_saved_encounter_ids = (
    validation_result[
        "null_encounter_ids"
    ]
)

null_saved_record_hashes = (
    validation_result[
        "null_record_hashes"
    ]
)

null_saved_processing_timestamps = (
    validation_result[
        "null_processing_timestamps"
    ]
)


print(
    f"Silver encounters row count: "
    f"{saved_encounter_count:,}"
)

print(
    f"Distinct encounter IDs: "
    f"{saved_distinct_encounter_count:,}"
)

print(
    f"Null encounter IDs: "
    f"{null_saved_encounter_ids:,}"
)

print(
    f"Null record hashes: "
    f"{null_saved_record_hashes:,}"
)

print(
    f"Null processing timestamps: "
    f"{null_saved_processing_timestamps:,}"
)


# Critical completion checks
if saved_encounter_count != 61459:

    raise ValueError(
        "Encounter row-count reconciliation failed. "
        f"Expected 61,459 but found "
        f"{saved_encounter_count:,}."
    )


if saved_distinct_encounter_count != 61459:

    raise ValueError(
        "Encounter ID uniqueness validation failed."
    )


if null_saved_encounter_ids != 0:

    raise ValueError(
        "Null encounter IDs were found "
        "in the saved Silver table."
    )


if null_saved_record_hashes != 0:

    raise ValueError(
        "Null record hashes were found "
        "in the saved Silver table."
    )


if null_saved_processing_timestamps != 0:

    raise ValueError(
        "Null processing timestamps were found "
        "in the saved Silver table."
    )


print(
    "Silver encounter-table verification passed."
)


# Display the saved schema and sample records
saved_silver_encounters.printSchema()

display(
    saved_silver_encounters.limit(10)
)


StatementMeta(, 44715bb7-8404-4d9c-8234-5c7979ac789b, 24, Finished, Available, Finished, False)

Silver encounters row count: 61,459
Distinct encounter IDs: 61,459
Null encounter IDs: 0
Null record hashes: 0
Null processing timestamps: 0
Silver encounter-table verification passed.
root
 |-- encounter_id: string (nullable = true)
 |-- encounter_start_timestamp: timestamp (nullable = true)
 |-- encounter_end_timestamp: timestamp (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- organization_id: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- payer_id: string (nullable = true)
 |-- encounter_class: string (nullable = true)
 |-- encounter_code: string (nullable = true)
 |-- encounter_description: string (nullable = true)
 |-- base_encounter_cost: decimal(18,2) (nullable = true)
 |-- total_claim_cost: decimal(18,2) (nullable = true)
 |-- payer_coverage: decimal(18,2) (nullable = true)
 |-- reason_code: string (nullable = true)
 |-- reason_description: string (nullable = true)
 |-- encounter_date: date (nullable = true)
 |-- encounter_duration_min

SynapseWidget(Synapse.DataFrame, a2c5b597-a552-4ee1-a7aa-4fe6216f83e6)